# Limpieza de datos — Simulador de Enfrentamiento Directo FIFA 2026

Notebook de **limpieza y preparacion del dataset** del proyecto *"Simulador de
Enfrentamiento Directo entre Selecciones de la Copa Mundial de la FIFA 2026 Basado en
Red Neuronal"*.

Esta preparado para **Google Colab**: se ejecuta celda por celda con el boton
*Ejecutar todo*. Toma el archivo comprimido `archive.zip` (descarga de Kaggle:
*FIFA World Cup 2026 Team Data*), verifica su contenido, diagnostica los problemas de
calidad del CSV y aplica las tareas de limpieza documentadas en el apartado **2.3
Limpieza de datos** del documento del proyecto.

**Resultado:** `teams_limpio.csv` (dataset depurado), `variables_modelo.csv`
(variables de entrada del simulador) y `codigos_equipos.csv` (diccionario de
equivalencias). Los dos primeros archivos no contienen ninguna letra: las
variables categoricas quedan codificadas como numeros enteros. Ambos alimentan el
pipeline de entrenamiento descrito en el apartado 2.2 del documento.

In [1]:
## Parte 1. Carga del paquete comprimido descargado de Kaggle
import os
import zipfile
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)

ZIP_NAME = "archive.zip"
# En Google Colab los archivos viven en /content; en local se usa la carpeta de trabajo.
ZIP_PATH = os.path.join("/content", ZIP_NAME) if os.path.isdir("/content") else ZIP_NAME

if not os.path.exists(ZIP_PATH):
    from google.colab import files
    print("Selecciona el archivo archive.zip en el dialogo que se abrira...")
    uploaded = files.upload()
    with open(ZIP_PATH, "wb") as f:
        f.write(uploaded[ZIP_NAME])

# Inventario del contenido del ZIP
with zipfile.ZipFile(ZIP_PATH) as z:
    archivos = z.namelist()
    print("Archivos contenidos en", ZIP_NAME, ":", archivos)
    z.extractall("data")

# Se lee el archivo CSV contenido en el paquete
CSV_NAME = [a for a in archivos if a.lower().endswith(".csv")][0]
df = pd.read_csv(os.path.join("data", CSV_NAME))

print("Archivo cargado:", CSV_NAME)
print("Forma del dataset (filas x columnas):", df.shape)
df.head()

Archivos contenidos en archive.zip : ['teams.csv']
Archivo cargado: teams.csv
Forma del dataset (filas x columnas): (48, 136)


,team,team_country,players_used,avg_age,possession,games,games_starts,minutes,minutes_90s,goals,assists,goals_assists,goals_pens,pens_made,pens_att,cards_yellow,cards_red,goals_per90,assists_per90,goals_assists_per90,goals_pens_per90,goals_assists_pens_per90,players_used_against,avg_age_against,possession_against,games_against,games_starts_against,minutes_against,minutes_90s_against,goals_against,...,gk_saves,gk_save_pct,gk_wins,gk_ties,gk_losses,gk_clean_sheets,gk_clean_sheets_pct,gk_pens_att,gk_pens_allowed,gk_pens_saved,gk_pens_missed,gk_pens_save_pct,gk_games_against,gk_games_starts_against,gk_minutes_against,gk_goals_against_against,gk_goals_against_per90_against,gk_shots_on_target_against_against,gk_saves_against,gk_save_pct_against,gk_wins_against,gk_ties_against,gk_losses_against,gk_clean_sheets_against,gk_clean_sheets_pct_against,gk_pens_att_against,gk_pens_allowed_against,gk_pens_saved_against,gk_pens_missed_against,gk_pens_save_pct_against
0,Algeria,Algeria,22,27.3,61.0,4,44,360,4.0,5,3,8,5,0,0,3,0,1.25,0.75,2.00,1.25,2.00,22,29.1,39.0,4,44,360,4.0,9,...,8,47.1,1,1,2,0,0.0,0,0,0,0,NaN,4,4,360,5,1.25,15,10,66.7,2,1,1,2,50.0,0,0,0,0,NaN
1,Argentina,Argentina,24,29.6,57.6,8,88,810,9.0,18,12,30,17,1,3,15,1,2.00,1.33,3.33,1.89,3.22,24,28.3,42.4,8,88,810,9.0,8,...,20,71.4,7,0,1,2,25.0,0,0,0,0,NaN,8,8,810,19,2.11,43,24,55.8,1,0,7,1,12.5,3,1,1,1,50.0
2,Australia,Australia,23,25.3,41.0,4,44,390,4.3,2,1,3,2,0,0,5,0,0.46,0.23,0.69,0.46,0.69,23,27.8,59.0,4,44,390,4.3,2,...,14,82.4,1,1,2,2,50.0,0,0,0,0,NaN,4,4,390,3,0.69,13,10,76.9,2,1,1,2,50.0,0,0,0,0,NaN
3,Austria,Austria,20,28.8,45.0,4,44,360,4.0,5,4,9,4,1,1,5,0,1.25,1.00,2.25,1.00,2.00,20,27.8,55.0,4,44,360,4.0,9,...,13,59.1,1,1,2,0,0.0,1,0,0,1,NaN,4,4,360,6,1.50,8,2,25.0,2,1,1,2,50.0,1,1,0,0,0.0
4,Belgium,Belgium,23,29.2,51.2,6,66,570,6.3,13,10,23,12,1,1,6,1,2.05,1.58,3.63,1.89,3.47,23,28.3,48.8,6,66,570,6.3,7,...,16,69.6,3,2,1,1,16.7,0,0,0,0,NaN,6,6,570,14,2.21,36,22,61.1,1,2,3,1,16.7,1,1,0,0,0.0


In [2]:
## Parte 1 (alternativa). Descarga directa desde Kaggle con kagglehub
# Alternativa al paso anterior: en lugar de subir el archivo, se descarga de Kaggle.
# Se deja comentada porque necesita la cuenta de Kaggle configurada en el entorno.
#
# !pip -q install kagglehub
# import kagglehub
# ruta = kagglehub.dataset_download("swaptr/fifa-wc-2026-teams")
# print("Dataset descargado en:", ruta)
print("Alternativa disponible: descomentar las lineas de kagglehub.")

Alternativa disponible: descomentar las lineas de kagglehub.


In [3]:
## Parte 1.1 / Conocer los datos
from pandas.api.types import is_string_dtype

print("Filas (selecciones):", df.shape[0])
print("Columnas (variables):", df.shape[1])
print("Variables de texto:", [c for c in df.columns if is_string_dtype(df[c])])
print("Variables numericas:", df.shape[1] - len([c for c in df.columns if is_string_dtype(df[c])]))

df[["team", "team_country", "games", "goals", "possession", "shots_on_target", "avg_age"]].head()

Filas (selecciones): 48
Columnas (variables): 136
Variables de texto: ['team', 'team_country']
Variables numericas: 134


,team,team_country,games,goals,possession,shots_on_target,avg_age
0,Algeria,Algeria,4,5,61.0,15,27.3
1,Argentina,Argentina,8,18,57.6,44,29.6
2,Australia,Australia,4,2,41.0,12,25.3
3,Austria,Austria,4,5,45.0,8,28.8
4,Belgium,Belgium,6,13,51.2,34,29.2


In [4]:
## Parte 2. Diagnostico de la calidad de los datos
hallazgos = []

# 2.1 Valores nulos por variable
nulos = df.isna().sum()
cols_nulos = nulos[nulos > 0]
hallazgos.append(("Variables con valores nulos", len(cols_nulos), ", ".join(cols_nulos.index)))

# 2.2 Variables constantes o totalmente vacias
constantes = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]
hallazgos.append(("Variables constantes o vacias", len(constantes), ", ".join(constantes)))

# 2.3 Duplicados
hallazgos.append(("Filas duplicadas", int(df.duplicated().sum()), "-"))
hallazgos.append(("Selecciones duplicadas", int(df["team"].duplicated().sum()), "-"))

# 2.4 Grupos de variables con valores identicos
numericas = df.select_dtypes(include=[np.number])
grupos = {}
for c in numericas.columns:
    clave = tuple(numericas[c].fillna(-999).round(6).tolist())
    grupos.setdefault(clave, []).append(c)
duplicadas = [g for g in grupos.values() if len(g) > 1]
hallazgos.append(("Grupos de variables duplicadas", len(duplicadas),
                  " | ".join(" = ".join(g) for g in duplicadas[:4]) + " | ..."))

# 2.5 Desbalance de partidos disputados
hallazgos.append(("Partidos disputados por seleccion",
                  f"{int(df['games'].min())} - {int(df['games'].max())}",
                  f"correlacion goals-games = {df['goals'].corr(df['games']):.2f}"))

diagnostico = pd.DataFrame(hallazgos, columns=["Hallazgo", "Cantidad", "Detalle"])
diagnostico

,Hallazgo,Cantidad,Detalle
0,Variables con valores nulos,6,"pens_won, pens_conceded, pens_won_against, pen..."
1,Variables constantes o vacias,6,"minutes_pct, minutes_pct_against, pens_won, pe..."
2,Filas duplicadas,0,-
3,Selecciones duplicadas,0,-
4,Grupos de variables duplicadas,18,players_used = players_used_against | games = ...
5,Partidos disputados por seleccion,3 - 8,correlacion goals-games = 0.89


In [5]:
## Parte 3. Limpieza de datos
df_clean = df.copy()

# 3.1 Eliminacion de variables constantes o totalmente vacias
constantes = [c for c in df_clean.columns if df_clean[c].nunique(dropna=True) <= 1]
df_clean = df_clean.drop(columns=constantes)
print("3.1 Tras eliminar constantes/vacias:", df_clean.shape)

# 3.2 Eliminacion de variables duplicadas (se conserva la primera de cada grupo)
numericas = df_clean.select_dtypes(include=[np.number])
grupos = {}
for c in numericas.columns:
    clave = tuple(numericas[c].fillna(-999).round(6).tolist())
    grupos.setdefault(clave, []).append(c)
a_eliminar = []
for g in grupos.values():
    if len(g) > 1:
        a_eliminar.extend(g[1:])          # se conserva la primera del grupo
df_clean = df_clean.drop(columns=[c for c in a_eliminar if c in df_clean.columns])
print("3.2 Tras eliminar duplicadas:", df_clean.shape)

# 3.3 Eliminacion de team_country (alias de team en 5 selecciones)
df_clean = df_clean.drop(columns=["team_country"])
print("3.3 Tras eliminar team_country:", df_clean.shape)

# 3.4 Imputacion de los valores nulos restantes con la mediana
nulos = df_clean.isna().sum()
cols_nulos = nulos[nulos > 0].index.tolist()
for c in cols_nulos:
    df_clean[c] = df_clean[c].fillna(df_clean[c].median())
print("3.4 Variables imputadas:", cols_nulos)
print("Valores nulos restantes:", int(df_clean.isna().sum().sum()))

3.1 Tras eliminar constantes/vacias: (48, 130)
3.2 Tras eliminar duplicadas: (48, 106)
3.3 Tras eliminar team_country: (48, 105)
3.4 Variables imputadas: ['gk_pens_save_pct', 'gk_pens_save_pct_against']
Valores nulos restantes: 0


In [6]:
## Parte 3.1 / Normalizacion por cantidad de partidos disputados
# Cada seleccion disputo entre 3 y 8 partidos, por lo que las metricas acumuladas
# no son comparables: se calculan ratios por partido y por 90 minutos.
df_clean["goals_per_game"] = (df_clean["goals"] / df_clean["games"]).round(3)
df_clean["shots_on_target_per_game"] = (df_clean["shots_on_target"] / df_clean["games"]).round(3)

comparacion = pd.DataFrame({
    "equipo": df_clean["team"],
    "goles (acumulado)": df_clean["goals"],
    "goles por partido": df_clean["goals_per_game"],
    "disparos a puerta (acumulado)": df_clean["shots_on_target"],
    "disparos por partido": df_clean["shots_on_target_per_game"],
})
comparacion.sort_values("goles por partido", ascending=False).head(10)

,equipo,goles (acumulado),goles por partido,disparos a puerta (acumulado),disparos por partido
19,Germany,11,2.750,28,7.000
29,Netherlands,10,2.500,22,5.500
17,England,20,2.500,53,6.625
18,France,20,2.500,59,7.375
38,Senegal,10,2.500,23,5.750
1,Argentina,18,2.250,44,5.500
4,Belgium,13,2.167,34,5.667
6,Brazil,10,2.000,30,6.000
31,Norway,12,2.000,29,4.833
24,Japan,8,2.000,13,3.250


In [7]:
## Parte 4. Codificacion de variables categoricas
# Tras la limpieza solo queda una variable de texto: el nombre de la seleccion.
# El modelo no admite letras, por lo que cada nombre se sustituye por un codigo
# entero y el diccionario de equivalencias se conserva en un archivo aparte.
codigos = {nombre: i + 1 for i, nombre in enumerate(sorted(df_clean["team"].unique()))}
df_clean["team_codigo"] = df_clean["team"].map(codigos)

diccionario = pd.DataFrame(sorted(codigos.items()), columns=["team", "team_codigo"])
print("Codigos generados:", len(codigos))
diccionario.head(10)

Codigos generados: 48


,team,team_codigo
0,Algeria,1
1,Argentina,2
2,Australia,3
3,Austria,4
4,Belgium,5
5,Bosnia–Herz,6
6,Brazil,7
7,Cabo Verde,8
8,Canada,9
9,Colombia,10


In [8]:
## Parte 4.1 / One-hot encoding y eliminacion del texto
# One-hot encoding: se agregan columnas binarias (0/1) para las categoricas con
# pocos valores distintos. Tras la limpieza no queda ninguna ademas del nombre
# de la seleccion, que se mantiene codificado y no expandido para no crear 48
# columnas adicionales.
bajas = [c for c in df_clean.columns
         if is_string_dtype(df_clean[c]) and c != "team" and df_clean[c].nunique() <= 20]
if bajas:
    df_clean = pd.get_dummies(df_clean, columns=bajas, drop_first=True)
    print("Variables codificadas en one-hot:", bajas)
else:
    print("No quedan categoricas de baja cardinalidad: no se aplica one-hot.")

# El nombre de la seleccion ya esta codificado, asi que se retira del dataset
df_clean = df_clean.drop(columns=["team"])
df_clean = df_clean[["team_codigo"] + [c for c in df_clean.columns if c != "team_codigo"]]

restantes = [c for c in df_clean.columns if is_string_dtype(df_clean[c])]
print("Variables de texto en el dataset depurado:", restantes)
print("Forma del dataset depurado:", df_clean.shape)

No quedan categoricas de baja cardinalidad: no se aplica one-hot.
Variables de texto en el dataset depurado: []
Forma del dataset depurado: (48, 107)


In [9]:
## Parte 5. Seleccion de variables para el simulador de enfrentamiento directo
# Las cuatro variables del vector de diferencias: goles, posesion, disparos a puerta
# y edad promedio del plantel.
FEATURES = ["goals", "possession", "shots_on_target", "avg_age"]

variables = ["team_codigo", "games"] + FEATURES + ["goals_per_game", "shots_on_target_per_game"]
df_modelo = df_clean[[v for v in variables if v in df_clean.columns]].copy()
df_modelo = df_modelo.sort_values("team_codigo").reset_index(drop=True)

print("Dataset para el modelo:", df_modelo.shape)
print("Variables de texto en el CSV del modelo:",
      [c for c in df_modelo.columns if is_string_dtype(df_modelo[c])])
print("Nulos:", int(df_modelo.isna().sum().sum()), "| Selecciones unicas:", df_modelo["team_codigo"].nunique())
df_modelo.head(10)

Dataset para el modelo: (48, 8)
Variables de texto en el CSV del modelo: []
Nulos: 0 | Selecciones unicas: 48


,team_codigo,games,goals,possession,shots_on_target,avg_age,goals_per_game,shots_on_target_per_game
0,1,4,5,61.0,15,27.3,1.250,3.750
1,2,8,18,57.6,44,29.6,2.250,5.500
2,3,4,2,41.0,12,25.3,0.500,3.000
3,4,4,5,45.0,8,28.8,1.250,2.000
4,5,6,13,51.2,34,29.2,2.167,5.667
5,6,4,4,46.0,14,26.6,1.000,3.500
6,7,5,10,53.0,30,29.2,2.000,6.000
7,8,4,4,37.3,12,30.8,1.000,3.000
8,9,5,8,54.4,31,26.9,1.600,6.200
9,10,5,5,57.8,30,30.0,1.000,6.000


In [10]:
## Parte 6. Exportacion de los datasets limpios
# Los archivos de datos no contienen ninguna letra: el unico archivo con texto
# es el diccionario de equivalencias, que sirve para interpretar los codigos.
df_clean.to_csv("teams_limpio.csv", index=False)
df_modelo.to_csv("variables_modelo.csv", index=False)
diccionario.to_csv("codigos_equipos.csv", index=False)

print("Archivos generados:")
for f in ["teams_limpio.csv", "variables_modelo.csv", "codigos_equipos.csv"]:
    print(" -", f, os.path.getsize(f), "bytes")

Archivos generados:
 - teams_limpio.csv 19000 bytes
 - variables_modelo.csv 1513 bytes
 - codigos_equipos.csv 578 bytes


In [11]:
## Parte 7. Resumen de la limpieza aplicada
resumen = pd.DataFrame([
    ["Variables del CSV original", df.shape[1], "-"],
    ["Variables del dataset depurado", df_clean.shape[1], "se conservan todas las utiles"],
    ["Variables del modelo", df_modelo.shape[1], ", ".join(df_modelo.columns)],
    ["Selecciones (filas)", df.shape[0], df_modelo.shape[0]],
    ["Valores nulos", int(df.isna().sum().sum()), int(df_modelo.isna().sum().sum())],
    ["Variables constantes o vacias", len(constantes), "eliminadas"],
    ["Variables duplicadas", len(set(a_eliminar)), "eliminadas"],
    ["Variables con nulos imputados", len(cols_nulos), ", ".join(cols_nulos)],
    ["Variables de texto", 1, "0 (el nombre queda como codigo entero)"],
], columns=["Aspecto", "Antes", "Despues"])
resumen

,Aspecto,Antes,Despues
0,Variables del CSV original,136,-
1,Variables del dataset depurado,107,se conservan todas las utiles
2,Variables del modelo,8,"team_codigo, games, goals, possession, shots_o..."
3,Selecciones (filas),48,48
4,Valores nulos,258,0
5,Variables constantes o vacias,6,eliminadas
6,Variables duplicadas,24,eliminadas
7,Variables con nulos imputados,2,"gk_pens_save_pct, gk_pens_save_pct_against"
8,Variables de texto,1,0 (el nombre queda como codigo entero)


### Criterio de limpieza aplicado

1. **Eliminar** toda variable constante o totalmente vacia, porque no aporta
   informacion y sesga el analisis descriptivo.
2. **Eliminar** las variables duplicadas, conservando la primera de cada grupo, para
   trabajar con un archivo sin redundancias.
3. **Eliminar** `team_country`, que solo repite el nombre de la seleccion con otra
   grafia (5 de las 48 selecciones usan un alias distinto).
4. **Imputar** con la mediana los valores nulos de las variables de penaltis del
   portero, que no tienen registro en 33 de las 48 selecciones.
5. **Normalizar** por cantidad de partidos disputados, porque las selecciones jugaron
   entre 3 y 8 partidos y las metricas acumuladas no son comparables entre si.
6. **Codificar** las variables categoricas: el nombre de la seleccion se sustituye
   por un codigo entero (`team_codigo`) y la equivalencia se guarda en
   `codigos_equipos.csv`, de modo que ni el dataset depurado ni el archivo del
   modelo contengan letras.
7. **Seleccionar** las cuatro variables del vector de diferencias (`goals`,
   `possession`, `shots_on_target`, `avg_age`) para el simulador de enfrentamiento
   directo descrito en el apartado 2.2 del documento.